In [ ]:
!ollama pull deepseek-r1:1.5b

In [ ]:
import os
import requests
from bs4 import BeautifulSoup
from dotenv import load_dotenv
from openai import OpenAI

In [ ]:
OLLAMA_BASE_URL = "http://localhost:11434/v1"

load_dotenv(override=True)
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')


**Scraper**

In [ ]:
def fetch_job_text(url):
    headers = {"User-Agent": "Mozilla/5.0"}

    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.text, "html.parser")

    # remove noise
    for tag in soup(["script", "style"]):
        tag.decompose()

    text = soup.get_text(separator=" ")

    return " ".join(text.split())

In [ ]:
system_prompt = """
You are an expert AI Job Analyst.

Your job:
- Analyze job posts clearly and structured
- Extract useful insights
- Be concise and practical

Return format:
1. Job Title
2. Required Skills
3. Experience Level
4. Salary clues (if any)
5. Recommendation (Apply or Skip with reason)

Rules:
- Be direct
- No fluff
- Use bullet points when needed
"""

In [ ]:
user_prompt = """Analyze this job post: {job_text}"""

In [ ]:
def analyze_job(job_text):
    response = ollama.chat.completions.create(
        model="deepseek-r1:1.5b",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt + (job_text)}
        ]
    )

    return response.choices[0].message.content

In [ ]:
url = "https://www.linkedin.com/jobs/view/4399966612/?eBP=CwEAAAGdkeiUgfPQXPJbsAzlxKn8TqSEGdm-PU6_HBfv5TaiVTrX-ZsnOiaFWFOMb4g1oe3Bvxc_Z0Ch2SU9_MwnZh4axmTR0tjnaJ97adnPwEHRSoldY0QiKoUtPu95va22G8HcSYyY-hm9xSNSie1XhilFwJ4qIiloGALkWBCwFG_yt2Tw8z0iuFKZJPeyskI-LyDfZs8m9MucSlWNVZijjc22mSGHfGyWZMDS-k4AZJLLy2WaXTqNGFlfIEkIT3vI1Fg9NJ4rftNLH56UNK2VjUAe0YWxpG-IncGfWMsMI0xQ40gyyc_0csN1B1LxgvPFL7gSxS7Pr9Sqjt2P6zHLxFW7ik5D1CFqKduGk1qzvclKYjSXRkaLKpP2I4F7U4P0Brmm9XN33pZeTdMpPmHDWk9ViDAoFBN59lpg7P158LH1QciWQs4PR-qVPfTsnZCqfgAtkHag9OwuvCtAfBV83lIhQIbx_TXk61qGTWP-LzImZm5eLtA4Tpj4y7l78A&trk=flagship3_search_srp_jobs&refId=XPQru462uw7noXT7ZifDSQ%3D%3D&trackingId=801zqVVlkUBQnJZjDHZtig%3D%3D"
print("\nScraping job post...\n")
job_text = fetch_job_text(url)

print("Analyzing with AI...\n")
result = analyze_job(job_text)

print(result)